In [ ]:
import mysql.connector
import pandas as pd

In [ ]:
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345",
    database="gaming")

cursor=conn.cursor()

# Deposit Table

In [ ]:
df_deposit =pd.read_csv("C:/Users/ANSH VERMA/Downloads/Vindita Data Analytics Project/Deposit Data.csv")

df_deposit.columns=[col.strip().replace(" ","_") for col in df_deposit.columns]

cursor.execute("Drop table if exists deposit")

cursor.execute("""create table deposit (user_id int,
                                      deposit_datetime varchar(255),
                                      deposit_amount int)""")

In [ ]:
for _, row in df_deposit.iterrows():
    cursor.execute("insert into deposit (user_id,deposit_datetime,deposit_amount) values(%s,%s,%s)", tuple(row))

conn.commit()
print("successfull")

successfull


# Withdrawal Table

In [ ]:
df_withdrawal=pd.read_csv("C:/Users/ANSH VERMA/Downloads/Vindita Data Analytics Project/Withdrawal_Data.csv")

df_withdrawal.columns=[col.strip().replace(" ","_") for col in df_withdrawal.columns]

cursor.execute("Drop table if exists withdrawal")

cursor.execute("""create table withdrawal (user_id int,
                                         withdrawal_datetime varchar(255),
                                         withdrawal_amount int)""")

In [ ]:
for _, row in df_withdrawal.iterrows():
    cursor.execute("insert into withdrawal (user_id,withdrawal_datetime,withdrawal_amount) values(%s,%s,%s)", tuple(row))

conn.commit()
print("successfull")

successfull


# Gameplay Table

In [ ]:
df_gameplay = pd.read_csv("C:/Users/ANSH VERMA/Downloads/Vindita Data Analytics Project/User Gameplay data.csv")

df_gameplay.columns=[col.strip().replace(" ","_") for col in df_gameplay.columns]

cursor.execute("drop table if exists gameplay")

cursor.execute("""create table gameplay (user_id int,
                                       games_played int,
                                       game_datetime varchar(255))""")


In [ ]:
for _, row in df_gameplay.iterrows():
    cursor.execute("insert into gameplay (user_id,games_played,game_datetime) values(%s,%s,%s)", tuple(row))

conn.commit()
print("successfull")


successfull


In [ ]:
df_gameplay

,User_ID,Games_played,game_datetime
0,851,1,01-10-2022 00:00
1,717,1,01-10-2022 00:00
2,456,1,01-10-2022 00:00
3,424,1,01-10-2022 00:00
4,845,1,01-10-2022 00:00
...,...,...,...
355261,658,1,31-10-2022 23:59
355262,582,1,31-10-2022 23:59
355263,272,1,31-10-2022 23:59
355264,563,1,31-10-2022 23:59


In [ ]:
df_withdrawal

,User_Id,Withdrawal_Datetime,Withdrawal_Amount
0,190,01-10-2022 00:03,5872
1,159,01-10-2022 00:16,9540
2,164,01-10-2022 00:24,815
3,946,01-10-2022 00:29,23000
4,763,01-10-2022 00:40,9473
...,...,...,...
3561,559,31-10-2022 23:27,5000
3562,407,31-10-2022 23:51,3000
3563,389,31-10-2022 23:56,14481
3564,11,31-10-2022 23:57,4000


In [ ]:
df_deposit

,User_Id,Deposit_Datetime,Deposit_Amount
0,357,01-10-2022 00:03,2000
1,776,01-10-2022 00:03,2500
2,492,01-10-2022 00:06,5000
3,803,01-10-2022 00:07,5000
4,875,01-10-2022 00:09,1500
...,...,...,...
17433,654,31-10-2022 23:57,1200
17434,980,31-10-2022 23:58,200
17435,2,31-10-2022 23:58,40000
17436,612,31-10-2022 23:58,2800


# Loyalty Points

In [ ]:
df_deposit.columns = [col.strip().lower().replace(' ', '_') for col in df_deposit.columns]
df_withdrawal.columns = [col.strip().lower().replace(' ', '_') for col in df_withdrawal.columns]
df_gameplay.columns = [col.strip().lower().replace(' ', '_') for col in df_gameplay.columns]


df_deposit['deposit_datetime'] = pd.to_datetime(df_deposit['deposit_datetime'], dayfirst=True)
df_withdrawal['withdrawal_datetime'] = pd.to_datetime(df_withdrawal['withdrawal_datetime'], dayfirst=True)
df_gameplay['game_datetime'] = pd.to_datetime(df_gameplay['game_datetime'], dayfirst=True)


In [ ]:
def classify_slot(dt):
    return 'S1' if dt.hour < 12 else 'S2'

df_deposit['slot'] = df_deposit['deposit_datetime'].apply(classify_slot)
df_withdrawal['slot'] = df_withdrawal['withdrawal_datetime'].apply(classify_slot)
df_gameplay['slot'] = df_gameplay['game_datetime'].apply(classify_slot)

In [ ]:
deposits_grouped = df_deposit.groupby('user_id').agg({
    'deposit_amount': 'sum',
    'deposit_datetime': 'count'
}).rename(columns={'deposit_amount': 'total_deposit', 'deposit_datetime': 'deposit_count'})

withdrawals_grouped = df_withdrawal.groupby('user_id').agg({
    'withdrawal_amount': 'sum',
    'withdrawal_datetime': 'count'
}).rename(columns={'withdrawal_amount': 'total_withdrawal', 'withdrawal_datetime': 'withdrawal_count'})

games_grouped = df_gameplay.groupby('user_id')['games_played'].sum().rename("total_games")

In [ ]:
loyalty = deposits_grouped.merge(withdrawals_grouped, how='outer', left_index=True, right_index=True)
loyalty = loyalty.merge(games_grouped, how='outer', left_index=True, right_index=True)
loyalty = loyalty.fillna(0)

In [ ]:
loyalty['loyalty_points'] = (
    0.01 * loyalty['total_deposit'] +
    0.005 * loyalty['total_withdrawal'] +
    0.001 * (loyalty['deposit_count'] - loyalty['withdrawal_count']).clip(lower=0) +
    0.2 * loyalty['total_games']
)

loyalty = loyalty.sort_values(by='loyalty_points', ascending=False)

In [ ]:
print("Top 10 users by loyalty points:")
print(loyalty.head(10))

Top 10 users by loyalty points:
         total_deposit  deposit_count  total_withdrawal  withdrawal_count  \
user_id                                                                     
634           515000.0            8.0        15737705.0              67.0   
99           1164800.0           47.0         2403141.0              15.0   
672          2158700.0           35.0          233750.0               5.0   
212          1924981.0           26.0          589850.0               4.0   
740          1738490.0           91.0          365288.0               7.0   
566          1819175.0           53.0          185071.0               3.0   
714          1676300.0           34.0               0.0               0.0   
421           878600.0           99.0         1269809.0              84.0   
369           650000.0           13.0         1586208.0               9.0   
30           1329000.0           51.0          152145.0               1.0   

         total_games  loyalty_points  
user

In [ ]:
loyalty.to_csv("loyalty_points_full.csv")


In [ ]:
import os
print(os.getcwd())


C:\Users\ANSH VERMA


In [ ]:
df_loyalty=pd.read_csv("C:/Users/ANSH VERMA/Downloads/Vindita Data Analytics Project/loyalty_points_full.csv")

In [ ]:
df_loyalty

,user_id,total_deposit,deposit_count,total_withdrawal,withdrawal_count,total_games,loyalty_points
0,634,515000.0,8.0,15737705.0,67.0,24,83843.325
1,99,1164800.0,47.0,2403141.0,15.0,10,23665.737
2,672,2158700.0,35.0,233750.0,5.0,10,22757.780
3,212,1924981.0,26.0,589850.0,4.0,1,22199.282
4,740,1738490.0,91.0,365288.0,7.0,2,19211.824
...,...,...,...,...,...,...,...
995,643,0.0,0.0,0.0,0.0,2,0.400
996,858,0.0,0.0,0.0,0.0,2,0.400
997,388,0.0,0.0,0.0,0.0,2,0.400
998,507,0.0,0.0,0.0,0.0,2,0.400


# Part A

## 1.Find Playerwise Loyalty points earned by Players in the following slots:-
    a. 2nd October Slot S1
    b. 16th October Slot S2
    b. 18th October Slot S1
    b. 26th October Slot S2

In [ ]:
targets = [
    ('2022-10-02', 'S1'),
    ('2022-10-16', 'S2'),
    ('2022-10-18', 'S1'),
    ('2022-10-26', 'S2')
]

results = []


for date_str, slot in targets:
    date_obj = pd.to_datetime(date_str)


    d = df_deposit[(df_deposit['deposit_datetime'].dt.date == date_obj.date()) & (df_deposit['slot'] == slot)]
    w = df_withdrawal[(df_withdrawal['withdrawal_datetime'].dt.date == date_obj.date()) & (df_withdrawal['slot'] == slot)]
    g = df_gameplay[(df_gameplay['game_datetime'].dt.date == date_obj.date()) & (df_gameplay['slot'] == slot)]


    d_group = d.groupby('user_id')['deposit_amount'].agg(['sum', 'count']).rename(columns={'sum': 'deposit_sum', 'count': 'deposit_count'})
    w_group = w.groupby('user_id')['withdrawal_amount'].agg(['sum', 'count']).rename(columns={'sum': 'withdrawal_sum', 'count': 'withdrawal_count'})
    g_group = g.groupby('user_id')['games_played'].sum().rename('total_games')


    combined = d_group.merge(w_group, how='outer', left_index=True, right_index=True)
    combined = combined.merge(g_group, how='outer', left_index=True, right_index=True)
    combined = combined.fillna(0)


    combined['loyalty_points'] = (
        0.01 * combined['deposit_sum'] +
        0.005 * combined['withdrawal_sum'] +
        0.001 * (combined['deposit_count'] - combined['withdrawal_count']).clip(lower=0) +
        0.2 * combined['total_games']
    )

    combined['date'] = date_str
    combined['slot'] = slot
    results.append(combined.reset_index())


final_loyalty_by_slot = pd.concat(results)


print("\n Player-wise Loyalty Points for Selected Date-Slot Combinations:")
print(final_loyalty_by_slot[['user_id', 'date', 'slot', 'loyalty_points']])


final_loyalty_by_slot.to_csv("C:/Users/ANSH VERMA/Downloads/loyalty_points_by_slot.csv", index=False)



 Player-wise Loyalty Points for Selected Date-Slot Combinations:
     user_id        date slot  loyalty_points
0          2  2022-10-02   S1           0.400
1          5  2022-10-02   S1           7.801
2          6  2022-10-02   S1          40.401
3          7  2022-10-02   S1           1.000
4          8  2022-10-02   S1           1.200
..       ...         ...  ...             ...
623      992  2022-10-26   S2         158.800
624      995  2022-10-26   S2           0.200
625      996  2022-10-26   S2           0.800
626      997  2022-10-26   S2           0.400
627      999  2022-10-26   S2           0.200

[2480 rows x 4 columns]


## 2. Calculate overall loyalty points earned and rank players on the basis of loyalty points in the month of October. In case of tie, number of games played should be taken as the next criteria for ranking.

In [ ]:



start_date = "2022-10-01"
end_date = "2022-10-31"

df_deposit_oct = df_deposit[(df_deposit['deposit_datetime'] >= start_date) & (df_deposit['deposit_datetime'] <= end_date)]
df_withdrawal_oct = df_withdrawal[(df_withdrawal['withdrawal_datetime'] >= start_date) & (df_withdrawal['withdrawal_datetime'] <= end_date)]
df_gameplay_oct = df_gameplay[(df_gameplay['game_datetime'] >= start_date) & (df_gameplay['game_datetime'] <= end_date)]


deposits_grouped = df_deposit_oct.groupby('user_id').agg({
    'deposit_amount': 'sum',
    'deposit_datetime': 'count'
}).rename(columns={'deposit_amount': 'total_deposit', 'deposit_datetime': 'deposit_count'})

withdrawals_grouped = df_withdrawal_oct.groupby('user_id').agg({
    'withdrawal_amount': 'sum',
    'withdrawal_datetime': 'count'
}).rename(columns={'withdrawal_amount': 'total_withdrawal', 'withdrawal_datetime': 'withdrawal_count'})

games_grouped = df_gameplay_oct.groupby('user_id')['games_played'].sum().rename("total_games")


loyalty = deposits_grouped.merge(withdrawals_grouped, how='outer', left_index=True, right_index=True)
loyalty = loyalty.merge(games_grouped, how='outer', left_index=True, right_index=True)
loyalty = loyalty.fillna(0)

loyalty['loyalty_points'] = (
    0.01 * loyalty['total_deposit'] +
    0.005 * loyalty['total_withdrawal'] +
    0.001 * (loyalty['deposit_count'] - loyalty['withdrawal_count']).clip(lower=0) +
    0.2 * loyalty['total_games']
)


loyalty_sorted = loyalty.sort_values(by=['loyalty_points', 'total_games'], ascending=[False, False]).reset_index()


loyalty_sorted['rank'] = loyalty_sorted.index + 1


print("\n Top 50 Users by Loyalty Points (October):")
print(loyalty_sorted[['rank', 'user_id', 'loyalty_points', 'total_games']].head(50))




 Top 50 Users by Loyalty Points (October):
    rank  user_id  loyalty_points  total_games
0      1      634       80843.460         23.0
1      2       99       23185.735         10.0
2      3      672       21857.578          9.0
3      4      212       21699.291          1.0
4      5      566       18652.554        177.0
5      6      740       18026.818          2.0
6      7      714       16154.233          6.0
7      8      421       15030.161       1508.0
8      9       30       13803.374         13.0
9     10      369       13646.105         36.0
10    11      587       13363.078        705.0
11    12      222       13348.803         10.0
12    13      352       12671.046        305.0
13    14      920       12527.800        892.0
14    15      162       12483.600          2.0
15    16      415       12304.215         25.0
16    17      569       12285.023         36.0
17    18      365       11597.745       3531.0
18    19      238       11524.437        877.0
19    20      78

## 3. What is the average deposit amount?

In [ ]:
while cursor.nextset():
    pass
cursor.execute("select round(avg(deposit_amount),2) as avg_deposit from deposit;")
result=cursor.fetchall()
print(result)

[(Decimal('5492.19'),)]


## 4. What is the average deposit amount per user in a month?

In [ ]:
while cursor.nextset():
    pass
cursor.execute("""SELECT
    AVG(user_total) AS avg_deposit_per_user
FROM (
    SELECT
        user_id,
        SUM(deposit_amount) AS user_total
    FROM deposit
    GROUP BY user_id
) AS monthly_user_totals;
;""")
result=cursor.fetchall()
print(result)

[(Decimal('104669.6492'),)]


## 5. What is the average number of games played per user?

In [ ]:
while cursor.nextset():
    pass
cursor.execute("""SELECT
    AVG(user_total_games) AS avg_games_per_user
FROM (
    SELECT
        user_id,
        SUM(games_played) AS user_total_games
    FROM gameplay
    GROUP BY user_id
) AS user_games;"""
)
result=cursor.fetchall()
print(result)

[(Decimal('355.2670'),)]


# Part B

## How bonus should be allocated to Top 50 leaderboard players from sum Rs.50,000 ?

To distribute bonus among top 50 players we can follow 2 approach which are widely used:-
1.Proportional Distribution
2.Bracket Method

## 1.Proportional Distribution

## 2.Bracket Method

## Conclusion :- Proportional Distribution is more balanced and suitable according to me here.

# Part C

## Would you say the loyalty point formula is fair or unfair?
## Can you suggest any way to make the loyalty point formula more robust?